In [8]:
from pathlib import Path
from typing import Final, Optional

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt
import polars as pl
from joblib import Memory

from src.config.constants import Product
from src.processing.dataset_spec import DatasetSpec
from src.processing.orderbook import OrderBookDataProcessor, OrderBookDataset
from src.research.black_scholes import BlackScholesCall

memory = Memory("./cache")

VELVETFRUIT_EXTRACT = Product.VELVETFRUIT_EXTRACT
VEV_OPTIONS_PREF: Final[str] = "VEV_"
ROUND: Final[int] = 3
DAY: Final[int] = 0

DATASET: Final[DatasetSpec] = DatasetSpec(ROUND, DAY)
ORDERBOOK_CSV_FILE: Final[Path] = DATASET.prices()
RAW_UNDERLYING_ORDERBOOK_DATA: Final[OrderBookDataProcessor] = OrderBookDataset(
    ORDERBOOK_CSV_FILE
).for_product(VELVETFRUIT_EXTRACT)
RAW_OPTIONS_ORDERBOOK_DATA: Final[OrderBookDataProcessor] = OrderBookDataset(
    ORDERBOOK_CSV_FILE
).for_product("VEV_")

underlying_orderbook_data = (
    RAW_UNDERLYING_ORDERBOOK_DATA.clean()
    .build()
    .select(["timestamp", "mid_price"])
    .rename({"mid_price": "spot_price"})
)
options_orderbook_data = (
    RAW_OPTIONS_ORDERBOOK_DATA.clean()
    .add_strikes_col()
    .add_microstructure_features()
    .add_price_features()
    .add_depth_features()
    .build()
).join(underlying_orderbook_data, on="timestamp", how="left")

BS: Final[BlackScholesCall] = BlackScholesCall(r=0, T_days=5)

In [9]:
options_orderbook_data

timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,strike_price,spread,imbalance,midprice_dev,log_returns,future_log_returns,microprice,microprice_dev,depth,spot_price
i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,i64,f64
0,"""VEV_5400""",22,25,null,null,null,null,24,25,null,null,null,null,23.0,5400,2,0.0,-235.83025,null,-3.828641,23.0,0.0,50,5250.0
0,"""VEV_6500""",0,16,null,null,null,null,1,16,null,null,null,null,0.5,6500,1,0.0,-258.33025,-3.828641,2.833213,0.5,0.0,32,5250.0
0,"""VEV_5500""",8,25,null,null,null,null,9,25,null,null,null,null,8.5,5500,1,0.0,-250.33025,2.833213,2.479993,8.5,0.0,50,5250.0
0,"""VEV_5200""",100,19,null,null,null,null,103,6,104,13,null,null,101.5,5200,3,0.0,-157.33025,2.479993,-0.649767,102.28,0.78,25,5250.0
0,"""VEV_5300""",52,6,51,19,null,null,54,25,null,null,null,null,53.0,5300,2,0.0,-205.83025,-0.649767,2.649781,52.387097,-0.612903,31,5250.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999900,"""VEV_5000""",247,23,null,null,null,null,253,10,254,13,null,null,250.0,5000,6,0.0,-8.83025,1.671313,1.604626,251.181818,1.181818,33,5244.0
999900,"""VEV_4000""",1234,14,1231,18,null,null,1254,14,1257,18,null,null,1244.0,4000,20,0.0,985.16975,1.604626,-2.026221,1244.0,0.0,28,5244.0
999900,"""VEV_5100""",162,23,null,null,null,null,166,10,167,13,null,null,164.0,5100,4,0.0,-94.83025,-2.026221,-5.793014,164.787879,0.787879,33,5244.0


In [10]:
options_orderbook_data = options_orderbook_data.with_columns(
    [(pl.col("spot_price") / pl.col("strike_price")).log().alias("log_moneyness")]
)

In [21]:
@memory.cache
def get_iv_vector() -> list[Optional[float]]:
    C_market_vector: npt.NDArray[np.float64] = options_orderbook_data[
        "microprice"
    ].to_numpy()
    K_vector: npt.NDArray[np.float64] = options_orderbook_data[
        "strike_price"
    ].to_numpy()
    S_vector: npt.NDArray[np.float64] = options_orderbook_data["spot_price"].to_numpy()

    IV_vector = [
        BS.implied_vol(C_market=c, K=k, S=s)
        for c, k, s in zip(C_market_vector, K_vector, S_vector)
    ]

    return IV_vector


IV_vector = get_iv_vector()

In [17]:
options_orderbook_data = options_orderbook_data.with_columns(
    pl.Series(name="IV", values=IV_vector)
).drop_nulls("IV")
options_orderbook_data

timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,strike_price,spread,imbalance,midprice_dev,log_returns,future_log_returns,microprice,microprice_dev,depth,spot_price,log_moneyness,IV
i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,f64,f64,f64,f64,f64,f64,i64,f64,f64,f64
0,"""VEV_5400""",22,25,null,null,null,null,24,25,null,null,null,null,23.0,5400,2,0.0,-235.83025,null,-3.828641,23.0,0.0,50,5250.0,-0.028171,0.303428
0,"""VEV_6500""",0,16,null,null,null,null,1,16,null,null,null,null,0.5,6500,1,0.0,-258.33025,-3.828641,2.833213,0.5,0.0,32,5250.0,-0.213574,0.677302
0,"""VEV_5500""",8,25,null,null,null,null,9,25,null,null,null,null,8.5,5500,1,0.0,-250.33025,2.833213,2.479993,8.5,0.0,50,5250.0,-0.04652,0.303375
0,"""VEV_5200""",100,19,null,null,null,null,103,6,104,13,null,null,101.5,5200,3,0.0,-157.33025,2.479993,-0.649767,102.28,0.78,25,5250.0,0.009569,0.305918
0,"""VEV_5300""",52,6,51,19,null,null,54,25,null,null,null,null,53.0,5300,2,0.0,-205.83025,-0.649767,2.649781,52.387097,-0.612903,31,5250.0,-0.009479,0.303469
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999900,"""VEV_5000""",247,23,null,null,null,null,253,10,254,13,null,null,250.0,5000,6,0.0,-8.83025,1.671313,1.604626,251.181818,1.181818,33,5244.0,0.047647,0.299366
999900,"""VEV_4000""",1234,14,1231,18,null,null,1254,14,1257,18,null,null,1244.0,4000,20,0.0,985.16975,1.604626,-2.026221,1244.0,0.0,28,5244.0,0.27079,0.423225
999900,"""VEV_5100""",162,23,null,null,null,null,166,10,167,13,null,null,164.0,5100,4,0.0,-94.83025,-2.026221,-5.793014,164.787879,0.787879,33,5244.0,0.027844,0.292519


In [18]:
options_orderbook_data["IV"].describe()

statistic,value
str,f64
"""count""",96017.0
"""null_count""",0.0
"""mean""",0.372564
"""std""",0.138282
"""min""",0.229798
"""25%""",0.292287
"""50%""",0.300403
"""75%""",0.431613
"""max""",1.123659


In [19]:
options_features_data = options_orderbook_data.select(
    ["timestamp", "strike_price", "spot_price", "mid_price", "log_moneyness", "IV"]
)
options_features_data

timestamp,strike_price,spot_price,mid_price,log_moneyness,IV
i64,i64,f64,f64,f64,f64
0,5400,5250.0,23.0,-0.028171,0.303428
0,6500,5250.0,0.5,-0.213574,0.677302
0,5500,5250.0,8.5,-0.04652,0.303375
0,5200,5250.0,101.5,0.009569,0.305918
0,5300,5250.0,53.0,-0.009479,0.303469
…,…,…,…,…,…
999900,5000,5244.0,250.0,0.047647,0.299366
999900,4000,5244.0,1244.0,0.27079,0.423225
999900,5100,5244.0,164.0,0.027844,0.292519


In [20]:
import plotly.graph_objects as go

df = options_features_data.sort("timestamp")

timestamps = df["timestamp"].unique().to_list()

# (optional but recommended if many timestamps)
timestamps = timestamps[::10]

# initial snapshot
t0 = timestamps[0]
snap0 = df.filter(pl.col("timestamp") == t0)
snap0 = snap0.filter(
    (pl.col("IV") > 0.005)
    & (pl.col("IV") < 1.0)
    & (pl.col("log_moneyness").abs() < 0.2)
)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=snap0["log_moneyness"].to_numpy(), y=snap0["IV"].to_numpy(), mode="markers"
    )
)

# build slider steps
steps = []
for t in timestamps:
    snap = df.filter(pl.col("timestamp") == t)

    steps.append(
        dict(
            method="update",
            args=[
                {"x": [snap["log_moneyness"].to_numpy()], "y": [snap["IV"].to_numpy()]}
            ],
            label=str(t),
        )
    )

# attach slider
fig.update_layout(
    sliders=[{"active": 0, "steps": steps}],
    xaxis_title="log moneyness",
    yaxis_title="IV",
    title="Vol Smile Over Time",
)

fig.show()